# Day 10: RAG with RAGWire

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

- Answer questions from your own documents, with citations that point at a file
- Today's corpus is real: the Alphabet and Amazon Form 10-K for the year ended 31 December 2024
- Watch the assistant refuse when the filings do not hold the answer
- Then hand retrieval to an agent, which decides when to search and how often
- Needs Qdrant running: `docker compose up -d` from the project folder

## 1. What RAG is

- Yesterday ended badly on purpose: a refund question against a baggage document still returned a chunk
- **Retrieve** the closest chunks, all of Day 9
- **Augment** the prompt by pasting them in as context
- **Generate** an answer that is only allowed to use them
- Step 3 catches yesterday's failure with one sentence in a prompt, nothing clever
- The same three steps today, over two real annual reports instead of a three-file sample

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv()

## 2. Check the database is up

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

client.get_collections()

- If that raised, run `docker compose up -d` in the project folder

## 3. Build the pipeline from config/day10_rag.yaml

In [ ]:
from ragwire import RAGWire

rag = RAGWire("config/day10_rag.yaml")

- One object: embedder, splitter, database and model, wired together by that file
- Every block in it is a day you have already done, and only the `llm` block needs a key

## 4. Ingest the documents, and re-run to see the skip

- Two filings sit in `filings/`, 197 pages between them
- `google-10-k-2024.pdf`, Alphabet Inc.
- `amazon-10-k-2024.pdf`, Amazon.com, Inc.
- The splitter is `strategy: page`: one chunk per PDF page, never sub-split, never merged
- A retrieved chunk is then a page a reader can open, which is what makes a citation checkable
- The model also reads each cover page and stores what it finds there as metadata
- The next cell embeds every page on your CPU and takes about three minutes, once

In [ ]:
stats = rag.ingest_directory("filings")

stats

- Two files in, 197 chunks out, one per page
- `metadata_failed: 0`: both cover pages were read
- Now run the **same cell again**

In [ ]:
rag.ingest_directory("filings")

- `processed` went to zero, `skipped` went up
- RAGWire fingerprints each file, so re-running does not duplicate chunks
- Duplicates crowd out the real answer when a retriever returns the same page three times
- Here is what the model pulled off the two cover pages:

## 5. Retrieve, without asking a model

In [ ]:
docs = rag.retrieve("What were Amazon's total net sales in 2024?")

for d in docs:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

- Plain LangChain `Document` objects, exactly what `similarity_search` gave you yesterday
- Each one is a whole page, and it knows which page it was
- No model was called here, and nothing was spent

## 6. Ask a question and read the citations

In [ ]:
answer = rag.query("What was Google Cloud's revenue in 2024, and how did it change from 2023?")

answer.text

- Whatever shape the answer takes, each figure carries a `[n]` marker back to the chunk it came from

In [ ]:
answer.citations

In [ ]:
answer.sources

- The answer points at the file it came from
- With a 108-page filing behind it, that pointer is the only practical way to check a number
- It is the difference between an assistant usable at work and a demo

## 7. When the documents do not have the answer

- Ask something a 10-K does not cover
- Amazon sells Kindles, so this sounds like a fair question to put to Amazon's annual report

In [ ]:
kindle = rag.query("What is the price of a Kindle Paperwhite?")

kindle.text

In [ ]:
kindle.refused

- It refused
- Now the important part: retrieval did **not** fail, and did not come back empty

In [ ]:
for d in kindle.documents:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

- Three chunks came back, all from Amazon's filing, none of them about the price of anything
- A vector search always returns its nearest neighbours, exactly as yesterday predicted
- Retrieval was as wrong today as it was yesterday. The difference is what happened next

## 8. The grounding rule that produces the refusal

In [ ]:
from ragwire.generation import DEFAULT_SYSTEM_PROMPT

print(DEFAULT_SYSTEM_PROMPT)

- Rule 3 is the whole day
- Without it, three chunks of Amazon boilerplate plus a price question gets a fluent paragraph, probably with a number in it
- With it, not answering is an allowed, correct outcome

In [ ]:
from ragwire.generation import REFUSAL_SENTINEL

REFUSAL_SENTINEL

- That string is what the model emits when the context does not support an answer
- Spotting it sets `answer.refused`: a magic string and an `if`
- A real boolean is what makes it usable in code: `if answer.refused: hand_off_to_a_human()`

## 9. Tune retrieval with top_k

- `top_k` decides how many chunks get pasted into the prompt
- The first setting people reach for, and it cuts both ways
- This question needs **both** filings to answer properly

In [ ]:
question = "What risks do Alphabet and Amazon describe about artificial intelligence?"

narrow = rag.retrieve(question, top_k=1)

for d in narrow:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

In [ ]:
wide = rag.retrieve(question, top_k=5)

for d in wide:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

- One page reaches one company. Five reach both
- Too low: a question needing two documents gets one, and the answer is half right or refuses wrongly
- Too high: unrelated pages get pasted in, you pay for them, and the model gets more chances to pick up the wrong thread
- Page chunks make this sharper than yesterday's 600-character ones. A whole page is one vector, so a single figure inside it is easy to miss, and `top_k: 5` in the config is why section 6 found its number at all

In [ ]:
both = rag.query(question)

both.text

In [ ]:
print(both.text)

In [ ]:
print(both.formatted())

In [ ]:
both.citations

In [ ]:
both.sources

### When the citations come back empty

- Both filings are cited above. It does not always go that way
- An empty `citations` means the model wrote no `[n]` marker, not that retrieval failed
- `sources` is derived from the markers the model actually wrote
- `answer.documents` always holds what was retrieved, marker or no marker

In [ ]:
for d in both.documents:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

- Those are the chunks the answer was built from
- Questions that span documents are the ones that expose a `top_k` set too low

## 10. Every field on the Answer object

- One fresh call, so everything below describes this answer and not one from four sections ago

In [ ]:
answer = rag.query("What were Alphabet's total revenues in 2024? Answer in one sentence.")

answer.text

In [ ]:
answer

In [ ]:
answer.to_dict().keys()

- Behind an API a caller wants text, sources and the refusal flag as separate fields, not one blob of prose
- One small trap:

In [ ]:
answer.formatted

- `formatted` is a **method**, not a property
- Printing it without the brackets gives you a bound-method repr

In [ ]:
print(answer.formatted())

## 11. An agent that decides when to search

- `rag.query()` retrieves exactly once, over the whole collection, then answers
- A question about two companies needs two searches, and nothing in `query()` can do that
- `auto_filter` is off in the config, so filtering is nobody's job yet
- Two tools fix both: `get_filter_context` reports what metadata the collection holds, `search_documents` runs retrieval with the filters the agent chose
- Day 7 turned a function into a tool, Day 8 handed the tool to an agent. Retrieval is just another tool

In [ ]:
from typing import Optional

from langchain.tools import tool


@tool
def get_filter_context(query: str) -> str:
    """Get available metadata fields, stored values, and filter suggestions for a query.

    Call this before search_documents when the query names a company, a form
    type or a year. Use what it returns to decide the filters.
    """
    return rag.get_filter_context(query)


@tool
def search_documents(query: str, filters: Optional[dict] = None) -> str:
    """Search the filings and return the matching pages.

    Args:
        query: The search query
        filters: Optional metadata filters decided from get_filter_context.
                 Pass {} or omit to search the whole collection.
    """
    results = rag.retrieve(query, top_k=3, filters=filters)
    if not results:
        return "No relevant documents found."

    pages = []
    for doc in results:
        source = doc.metadata["file_name"] + ", page " + str(doc.metadata["page_number"])
        pages.append("[" + source + "]\n" + doc.page_content)

    return "\n\n---\n\n".join(pages)

- The docstrings are the whole interface, as on Day 7: they are how the agent learns this collection has filters at all
- `search_documents` labels every page with its file and page number, so the answer can point at something you can open
- Run the filter tool by hand first, so the agent's first move is not a mystery:

In [ ]:
print(get_filter_context.invoke(
    {"query": "How did Google Cloud revenue change in 2024?"}
))

- Stored values on top, the filters it suggests for this question underneath
- The agent reads this text and decides. Nothing here applies a filter by itself

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.messages import HumanMessage
from langchain_openrouter import ChatOpenRouter
from langgraph.checkpoint.memory import InMemorySaver

model = ChatOpenRouter(model="qwen/qwen3.8-27b", temperature=0)

agent = create_agent(
    model=model,
    tools=[get_filter_context, search_documents],
    system_prompt="""You answer questions about SEC filings, using the tools only.
Call get_filter_context first when the question names a company, a form type or a year.
Then call search_documents with the filters you chose from the values it reported.
A question about two companies needs one search per company.
Say so if the filings do not answer the question.""",
    middleware=[ModelCallLimitMiddleware(run_limit=8, exit_behavior="end")],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "filings"}}

- The call limit is Day 8's: an agent that cannot find a number will keep looking for it, and a 10-K has plenty it does not state
- The checkpointer is Day 6's, so the follow-up question below can say "that" and mean something

- Day 8's trace again: `.invoke` hands back the whole conversation, and every tool use is a message in it
- An AI message carries `tool_calls`, what the agent asked for
- The `ToolMessage` under it carries the result, the text the model actually read
- Results are truncated here: three chunks of a 10-K run to thousands of characters

- One question, two companies:

In [ ]:
question = "Which company had higher total revenue in 2024, Alphabet or Amazon?"

result = agent.invoke({"messages": [HumanMessage(question)]}, config=config)


In [ ]:
result

In [ ]:
result['messages'][-1].pretty_print()

- The agent asked what the collection holds, then searched with a filter it chose
- Each `tool_result` is a whole page of filing text, labelled with the file and page it came from
- `rag.query()` could not have done this: one retrieval, no filters, one answer

- Now a follow-up that only makes sense if the previous turn is still there:

In [ ]:
follow = agent.invoke(
    {"messages": [HumanMessage("How much of Alphabet's revenue came from Google Cloud?")]},
    config=config,
)

In [ ]:
follow

In [ ]:
follow['messages'][-1].pretty_print()

- Nothing printed: no tool call, no tool result
- The chunks from the previous turn are still in the thread, so the agent answered from them
- That is the checkpointer doing its job, and also a warning: an agent with memory can answer without checking anything

- An agent, a retrieval tool, a grounding prompt and a memory: Days 6 to 10, in one object